In [ ]:
import json
import re
from pathlib import Path

import igraph
import pandas as pd

PROCESSED_DATA_PATH = Path("bpa_processed.csv")
RESOURCE_DIR = Path("res")


def normalize_bus_name(value):
    """Normalize a bus name and reject failed endpoint extractions."""
    if pd.isna(value):
        return None

    bus_name = re.sub(r"\s+", " ", str(value).strip()).upper()
    if not bus_name or bus_name == "---":
        return None
    return bus_name


# Load the processed BPA transmission-line records.
bpa = pd.read_csv(
    PROCESSED_DATA_PATH,
    usecols=["Name", "First Bus", "Second Bus"],
    low_memory=False,
)

line_data = bpa.copy()
line_data["First Bus"] = line_data["First Bus"].map(normalize_bus_name)
line_data["Second Bus"] = line_data["Second Bus"].map(normalize_bus_name)

valid_endpoints = line_data["First Bus"].notna() & line_data["Second Bus"].notna()
invalid_endpoint_count = int((~valid_endpoints).sum())
line_data = line_data.loc[valid_endpoints].reset_index(drop=True)

# Construct the graph as in the NYISO notebook: buses are vertices and
# each unique unordered endpoint pair is one transmission-line edge.
bus_names = sorted(set(line_data["First Bus"]) | set(line_data["Second Bus"]))
bus_name_to_index = {bus_name: index for index, bus_name in enumerate(bus_names)}
index_to_bus_name = {index: bus_name for bus_name, index in bus_name_to_index.items()}

edge_set = set()
for first_bus, second_bus in zip(line_data["First Bus"], line_data["Second Bus"]):
    first_index = bus_name_to_index[first_bus]
    second_index = bus_name_to_index[second_bus]
    edge_set.add(tuple(sorted((first_index, second_index))))

g = igraph.Graph(n=len(bus_names), directed=False)
g.add_edges(sorted(edge_set))
g.vs["name"] = bus_names

# Save the graph and mappings in the BPA resource directory.
RESOURCE_DIR.mkdir(parents=True, exist_ok=True)
g.write_pickle(RESOURCE_DIR / "outage_graph.pkl")
(RESOURCE_DIR / "bus_name_to_index.json").write_text(
    json.dumps(bus_name_to_index, indent=2, sort_keys=True)
)
(RESOURCE_DIR / "index_to_bus_name.json").write_text(
    json.dumps(index_to_bus_name, indent=2, sort_keys=True)
)

print(f"Loaded outage records: {len(bpa):,}")
print(f"Usable outage records: {len(line_data):,}")
print(f"Excluded records without two usable endpoints: {invalid_endpoint_count:,}")
print(f"Number of buses: {g.vcount():,}")
print(f"Number of unique lines: {g.ecount():,}")
print(f"Graph is connected: {g.is_connected()}")

Loaded outage records: 54,399
Usable outage records: 53,945
Excluded records without two usable endpoints: 454
Number of buses: 1,237
Number of unique lines: 1,562
Graph is connected: False


In [2]:
# Count the transmission lines in every connected subgraph.
connected_components = g.connected_components()

component_rows = []
for component_id, vertex_indices in enumerate(connected_components):
    connected_subgraph = g.subgraph(vertex_indices)
    component_rows.append(
        {
            "component_id": component_id,
            "num_buses": connected_subgraph.vcount(),
            "num_lines": connected_subgraph.ecount(),
        }
    )

component_summary = (
    pd.DataFrame(component_rows)
    .sort_values(["num_lines", "num_buses"], ascending=False)
    .reset_index(drop=True)
)
component_summary.insert(0, "size_rank", component_summary.index + 1)

# Every graph bus and line must belong to exactly one connected subgraph.
assert component_summary["num_buses"].sum() == g.vcount()
assert component_summary["num_lines"].sum() == g.ecount()

print(f"Number of connected subgraphs: {len(component_summary):,}")
print(
    "Lines accounted for across all connected subgraphs: "
    f"{component_summary['num_lines'].sum():,}"
)
component_summary

Number of connected subgraphs: 58
Lines accounted for across all connected subgraphs: 1,562


,size_rank,component_id,num_buses,num_lines
0,1,0,1083,1463
1,2,20,14,14
2,3,21,7,6
3,4,30,6,5
4,5,16,5,4
5,6,29,5,4
6,7,22,4,3
7,8,37,4,3
8,9,8,3,2
9,10,19,3,2


In [ ]:
# Keep only outage records whose line belongs to the largest connected subgraph.
largest_component = max(connected_components, key=len)
largest_component_buses = {
    g.vs[vertex_index]["name"] for vertex_index in largest_component
}

# Reload every column so filtering does not discard non-graph attributes.
processed_bpa = pd.read_csv(PROCESSED_DATA_PATH, low_memory=False)
normalized_first_bus = processed_bpa["First Bus"].map(normalize_bus_name)
normalized_second_bus = processed_bpa["Second Bus"].map(normalize_bus_name)
is_in_largest_component = (
    normalized_first_bus.isin(largest_component_buses)
    & normalized_second_bus.isin(largest_component_buses)
)

filtered_bpa = processed_bpa.loc[is_in_largest_component].copy()
removed_record_count = len(processed_bpa) - len(filtered_bpa)

# Write atomically, replacing the processed CSV only after the new file is complete.
temporary_path = PROCESSED_DATA_PATH.with_suffix(".tmp.csv")
filtered_bpa.to_csv(temporary_path, index=False)
temporary_path.replace(PROCESSED_DATA_PATH)

print(f"Largest connected subgraph buses: {len(largest_component_buses):,}")
print(f"Kept outage records: {len(filtered_bpa):,}")
print(f"Removed outage records: {removed_record_count:,}")
print(f"Rewrote {PROCESSED_DATA_PATH}")